<a href="https://colab.research.google.com/github/Angelmzm/siteshow-dados/blob/main/siteshow_coleta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Parte 1: O Acesso à Página ( requests )

In [1]:
import requests
from bs4 import BeautifulSoup

# O link oficial que você escolheu
url = "https://www.sympla.com.br/eventos/campinas-sp"

# O "disfarce": dizemos ao site que somos um navegador comum no Windows
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# O robô tenta acessar a página usando o disfarce
resposta = requests.get(url, headers=headers)

# Verificamos se o acesso deu certo (Código 200 significa sucesso!)
if resposta.status_code == 200:
    print("Sucesso! Conseguimos conectar ao Sympla Campinas.")
else:
    print(f"Erro {resposta.status_code}: O site bloqueou a requisição automática.")

Sucesso! Conseguimos conectar ao Sympla Campinas.


Parte 2: A Captura dos Dados ( BeautifulSoup )


In [2]:
# Se a resposta anterior foi um sucesso, vamos processar a página
if resposta.status_code == 200:
    # O BeautifulSoup lê todo o código HTML da página do Sympla
    sopa = BeautifulSoup(resposta.text, 'html.parser')

    # Criamos listas vazias para guardar o que o robô encontrar
    lista_nomes = []
    lista_datas = []
    lista_locais = []

    # O robô procura por todos os links de cartões de eventos na página
    # Nota: Usamos a estrutura padrão de cartões do ecossistema Sympla
    cartoes = sopa.find_all('a', class_=lambda x: x and 'EventCard' in x) or sopa.find_all('a')

    for card in cartoes:
        # Procuramos o título do evento dentro do cartão
        elemento_nome = card.find('h3') or card.find(class_=lambda x: x and 'title' in x.lower())

        if elemento_nome:
            nome = elemento_nome.text.strip()
            lista_nomes.append(nome)

            # Buscando a data (geralmente fica em um elemento de texto menor ou específico)
            elemento_data = card.find(class_=lambda x: x and 'date' in x.lower()) or card.find('span')
            data = elemento_data.text.strip() if elemento_data else "Data não informada"
            lista_datas.append(data)

            # Buscando o local (bairro, casa de show ou instituição)
            elemento_local = card.find(class_=lambda x: x and 'place' in x.lower()) or card.find('p')
            local = elemento_local.text.strip() if elemento_local else "Campinas - SP"
            lista_locais.append(local)

    print(f"Varredura concluída! O robô mapeou {len(lista_nomes)} possíveis eventos em Campinas.")
else:
    print("Execute a primeira célula com sucesso antes de rodar esta.")

Varredura concluída! O robô mapeou 16 possíveis eventos em Campinas.


Estruturando os dados com o Pandas

In [3]:
import pandas as pd

# Pegamos as listas criadas pelo robô e colocamos em um formato de dicionário
dados_brutos = {
    'Nome do Evento': lista_nomes,
    'Data/Horário': lista_datas,
    'Local': lista_locais
}

# O Pandas transforma esse dicionário em uma tabela organizada (DataFrame)
df_siteshow = pd.DataFrame(dados_brutos)

# Exibe a tabela na tela para checarmos o resultado
print("--- Tabela Estruturada do SiteShow (Dados Coletados) ---")
display(df_siteshow)


--- Tabela Estruturada do SiteShow (Dados Coletados) ---


,Nome do Evento,Data/Horário,Local
0,Kit Digital Turma da Monica Scrapbook,Data não informada,"Local a definir - São Paulo, SP"
1,47ª Quermesse do Centenário da Igreja do Calvário,Data não informada,"Igreja do Calvário - São Paulo, SP"
2,"Sessões de Skate, Patins e Scooter na Pista do...",Data não informada,Farol Santander Pista do 21 Rajas Skatepark - ...
3,ARENA STAR LASER,Data não informada,"Tietê Plaza Shopping - São Paulo, SP"
4,Red Flame - Tributo ao Simply Red 29 agosto ...,Data não informada,"Teatro PEN LIFE - São Bernardo do Campo, SP"
5,Queen Music Tribute 16 Agosto Domingo 18h,Data não informada,"Teatro PEN LIFE - São Bernardo do Campo, SP"
6,"Tributo Bon Jovi Bavini Sábado, 12 de setembr...",Data não informada,"Teatro PEN LIFE - São Bernardo do Campo, SP"
7,"A Música dos GAMES, com a MOV Orquestra 11 ju...",Data não informada,"Teatro PEN LIFE - São Bernardo do Campo, SP"
8,K-Pop Odyssey – A Maior Experiência do K-Pop A...,Data não informada,"Teatro PEN LIFE - São Bernardo do Campo, SP"
9,Tributo a Andrea Boccelli Sexta 10 julho às ...,Data não informada,"Teatro PEN LIFE - São Bernardo do Campo, SP"


Salvando a Tabela em um Arquivo

In [4]:
# Salvando a tabela em um arquivo CSV chamado 'eventos_campinas.csv'
# O 'index=False' serve para não criar uma coluna de números desnecessária
df_siteshow.to_csv('eventos_campinas.csv', index=False, encoding='utf-8-sig')

print("Arquivo 'eventos_campinas.csv' gerado com sucesso!")

Arquivo 'eventos_campinas.csv' gerado com sucesso!
